# Test-set addendum: posterior-mean ensemble

**Scope: test set, post-hoc addendum.** It extends the frozen test protocol after its pre-registered results (`docs/PROTOCOL.md` §9). It is specified in `docs/PROTOCOL.md` §11 (Addendum A), frozen and signed off before its test run. It is a **post-hoc test confirmation**, separate from the pre-registered §3 table.

CPU only (like notebook 03): never touches the GPU.

---
# Part 1: weighted posterior-mean ensemble (E3w) on the 100 test images

**Method (Addendum A; identical to notebook 04 Check 13).** Per test image, correct label, 5 frozen-CSGM members:
- restarts r0, r1: the saved CSGM test runs;
- shifts (0,1), (1,0), (1,1): CSGM on y circularly shifted by that many LR pixels, started from the r0 initialization (`scripts/run_sr4_ensemble_test.py`), output shifted back by 4 HR pixels per LR pixel.

Ensembles, each followed by back-projection (BP):
- **E3w (primary):** weights wₖ ∝ exp(−(Lₖ − L_min)/std(L)), Lₖ = LR loss ‖y − A xₖ‖² of member k in original coordinates;
- E3: uniform mean of the 5; E1: mean of r0, r1; E2: mean of r0 + 3 shifts.

Reference: **CSGM + BP** as frozen (the restart with the lowest final LR loss), recomputed here and checked against `csgm_test_postproc_v1/`.

**Primary tests (Holm over 2):** E3w + BP vs CSGM + BP on PSNR and LPIPS, paired over the 100 images (same code as notebook 03 Part 3).

## Setup

In [ ]:
import os
import sys
from pathlib import Path

os.environ["JAX_PLATFORMS"] = "cpu"  # analysis only: never touch the GPU
SCRIPTS = Path.home() / "work" / "IADrift" / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import csv
import hashlib
import io
import json
import math

import jax
import jax.numpy as jnp
import lpips
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display
from jax.scipy.sparse.linalg import cg
from scipy import stats

from analyze_sr4_csgm_test import psnr_of_mean_mse
from freq_metrics import BANDS, band_snr_db, blockiness
from plot_sr4_csgm_test import DATA_DIR, load_hr256

%matplotlib inline

BAND_NAMES = [name for name, *_ in BANDS]
CLASS_INDEX = json.loads((DATA_DIR / "imagenet_class_index.json").read_text())


def class_name(label):
    return CLASS_INDEX[str(int(label))][1].replace("_", " ")


def A_sr4(x):
    # Identical to run_sr4_csgm_test.A_sr4 (antialiased bicubic 256 -> 64).
    return jax.image.resize(x.astype(jnp.float32), (x.shape[0], 64, 64, 3), method="cubic", antialias=True)


_A_transpose = jax.linear_transpose(A_sr4, jnp.zeros((1, 256, 256, 3), jnp.float32))


@jax.jit
def _backproject(x, y):
    # Same as notebook 03 Part 2 (CG, tol 1e-6, maxiter 500), so CSGM + BP reproduces csgm_test_postproc_v1.
    def AT(v):
        return _A_transpose(v)[0]
    u, _ = cg(lambda v: A_sr4(AT(v)), y - A_sr4(x), tol=1e-6, maxiter=500)
    return x + AT(u)


def backproject(x, y):
    return np.asarray(_backproject(jnp.asarray(x, jnp.float32)[None], jnp.asarray(y, jnp.float32)[None])[0])


def lr_loss(x, y):
    return float(jnp.sum((A_sr4(jnp.asarray(x, jnp.float32)[None]) - jnp.asarray(y, jnp.float32)[None]) ** 2))


def psnr_db(hr, x):
    mse = np.mean(((np.asarray(hr, np.float64) - np.asarray(x, np.float64)) / 2.0) ** 2)
    return float(-10 * np.log10(mse))


def mse(hr, x):
    return float(np.mean((np.asarray(hr, np.float64) - np.asarray(x, np.float64)) ** 2))


torch.set_num_threads(4)
LPIPS_NET = lpips.LPIPS(net="alex", verbose=False).eval()


def lpips_dist(x, ref):
    to_t = lambda a: torch.from_numpy(np.ascontiguousarray(np.clip(a, -1, 1).transpose(2, 0, 1)[None])).float()
    with torch.no_grad():
        return float(LPIPS_NET(to_t(x), to_t(ref)))


def evaluate(x, hr, y):
    return {"psnr": psnr_db(hr, x), "psnr_clipped": psnr_db(hr, np.clip(x, -1, 1)), "lpips": lpips_dist(x, hr),
            "blockiness": blockiness(x), "lr_loss": lr_loss(x, y), "band_snr_db": band_snr_db(x, hr)}


def paired_stats(a, b, key, table):
    # Notebook 03 Part 3 code: a - b over the images; for LPIPS / blockiness lower is better.
    d = table(a, key) - table(b, key)
    lower_better = key in ("lpips", "blockiness")
    half = 1.96 * d.std(ddof=1) / np.sqrt(len(d))
    return {"mean": d.mean(), "lo": d.mean() - half, "hi": d.mean() + half,
            "p": stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0,
            "better": int(np.sum(d < 0 if lower_better else d > 0)), "n": len(d)}


def holm(pvalues):
    order = np.argsort(pvalues)
    adjusted, running = np.empty(len(pvalues)), 0.0
    for rank, k in enumerate(order):
        running = max(running, min(1.0, (len(pvalues) - rank) * pvalues[k]))
        adjusted[k] = running
    return adjusted


CROP = 80
show_img = lambda x: (np.clip(x, -1, 1) + 1) / 2


def detail_crop(hr):
    # Notebook 03: the 80x80 window (16-pixel grid) with the most HR gradient energy.
    g = np.abs(np.diff(hr.mean(-1), axis=0))[:, :-1] + np.abs(np.diff(hr.mean(-1), axis=1))[:-1, :]
    pos = [(r, c) for r in range(0, 256 - CROP, 16) for c in range(0, 256 - CROP, 16)]
    return max(pos, key=lambda p: g[p[0]:p[0] + CROP, p[1]:p[1] + CROP].sum())


def side_by_side(i, hr, title, shown):
    # shown: [(name, image, metrics)]; HR + panels, with an 80x80 detail crop row.
    r, c0 = detail_crop(hr)
    fig, axes = plt.subplots(2, len(shown) + 1, figsize=(4 * (len(shown) + 1), 8.6))
    axes[0, 0].imshow(show_img(hr))
    axes[0, 0].add_patch(plt.Rectangle((c0, r), CROP, CROP, fill=False, color="yellow", linewidth=1.5))
    axes[0, 0].set_title(title, fontsize=9)
    axes[1, 0].imshow(show_img(hr[r:r + CROP, c0:c0 + CROP]), interpolation="nearest")
    axes[1, 0].set_title("HR (crop)", fontsize=9)
    for k, (m, x, mm) in enumerate(shown, start=1):
        axes[0, k].imshow(show_img(x))
        axes[0, k].set_title(f"{m}\nPSNR {mm['psnr']:.2f} dB | LPIPS {mm['lpips']:.3f} | block. {mm['blockiness']:.2f}", fontsize=8)
        axes[1, k].imshow(show_img(x[r:r + CROP, c0:c0 + CROP]), interpolation="nearest")
        axes[1, k].set_title(f"{m} (crop)", fontsize=9)
    for ax in axes.flat:
        ax.axis("off")
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.9, wspace=0.04, hspace=0.2)
    plt.show()


print("JAX backend:", jax.default_backend())

## Part 1, step 1: load and validate the runs

Checks: the manifest hash matches the frozen plan (`ca2d0ec5e6ff`, PROTOCOL §11), all 300 shifted runs are saved, finite and in the plan with the frozen settings, and the runner's LR loss of each shifted-back member matches the one recomputed here.

In [ ]:
CSGM_STEM = "phase1_sr4_csgm_label_test_v2_n100_9e24c78a47ab"
ENS_RUN_DIR = DATA_DIR / "csgm_test_ensemble_v1"
ENS_POST_DIR = DATA_DIR / "csgm_test_ensemble_postproc_v1"
ENS_POST_DIR.mkdir(exist_ok=True)
FROZEN_HASH = "ca2d0ec5e6ff"
SHIFTS = [(0, 1), (1, 0), (1, 1)]

manifest = json.loads((ENS_RUN_DIR / "manifest.json").read_text())
spec = {k: v for k, v in manifest.items() if k != "hash"}
spec_hash = hashlib.sha256(json.dumps(spec, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()[:12]
assert manifest["hash"] == spec_hash == FROZEN_HASH, (manifest["hash"], spec_hash)
ens_rows = [json.loads(l) for l in (ENS_RUN_DIR / f"results_{FROZEN_HASH}.jsonl").open() if l.strip()]
plan_keys = {(r["image_index"], *r["shift"]): r for r in manifest["run_plan"]}
ens = {}
for r in ens_rows:
    p = plan_keys[(r["image_index"], *r["shift"])]
    for f in ("learning_rate", "num_steps", "class_label", "start_restart"):
        assert r[f] == p[f], (r["image_index"], r["shift"], f)
    assert math.isfinite(r["final_loss"]) and math.isfinite(r["lr_loss_unshifted"])
    ens[(r["image_index"], *r["shift"])] = r
assert len(ens) == len(ens_rows), "duplicate runs"

test_sample = [int(i) for i in np.load(DATA_DIR / "sr4_test_sample_100_seed2028.npz")["test_sample_indices"]]
images = [i for i in test_sample if all((i, *s) in ens for s in SHIFTS)]
complete = len(images) == len(test_sample) == 100

csgm_rows = [json.loads(l) for l in (DATA_DIR / f"{CSGM_STEM}.jsonl").open() if l.strip()]
csgm = {(r["image_index"], r["restart"]): r for r in csgm_rows if r["condition"] == "oracle"}
kept = {i: min((0, 1), key=lambda r: (csgm[(i, r)]["final_loss"], r)) for i in images}  # the frozen restart rule
true_label = {i: csgm[(i, 0)]["true_label"] for i in images}

print(f"Shifted runs saved: {len(ens_rows)}/{len(manifest['run_plan'])}; images complete: {len(images)}/{len(test_sample)}")
print(f"Mean time per run: {np.mean([r['seconds'] for r in ens_rows]):.0f} s")
if not complete:
    display(Markdown(f"**PRELIMINARY: only {len(images)} of {len(test_sample)} images are finished.**"))

## Part 1, step 2: build the ensembles, back-project, score (cached per image)

Per image: the 5 members, their LR losses Lₖ (original coordinates), E1/E2/E3/E3w and the frozen CSGM reference, each + BP; metrics as in notebook 03 Part 2. Also stored per image: the E3w weights and, for the MMSE check, each member's MSE to HR and the uniform mean's MSE (before BP).

In [ ]:
index_rows = {int(r["index"]): r for r in csv.DictReader(open(DATA_DIR / "imageNet_x4_index.csv"))}
lr_all = np.load(DATA_DIR / "imageNet_x4.npy", mmap_mode="r")
csgm_post = json.loads((DATA_DIR / "csgm_test_postproc_v1" / "metrics.json").read_text())
MEMBERS = ["r0", "r1"] + [f"shift{sy}{sx}" for sy, sx in SHIFTS]
ENSEMBLES = {"E1: mean of 2 restarts": ["r0", "r1"], "E2: mean of r0 + 3 shifts": ["r0"] + MEMBERS[2:],
             "E3: mean of 5 CSGM runs": MEMBERS, "E3w: weighted 5 CSGM runs": MEMBERS}
ENS_METHODS = [n + " + BP" for n in ENSEMBLES]


def load_members(i):
    out = {}
    for r in (0, 1):
        with np.load(DATA_DIR / f"{CSGM_STEM}_arrays" / f"img{i:05d}_oracle_r{r}.npz") as data:
            out[f"r{r}"] = data["x_hat"]
    for sy, sx in SHIFTS:
        with np.load(ENS_RUN_DIR / f"img{i:05d}_shift{sy}{sx}_r0.npz") as data:
            out[f"shift{sy}{sx}"] = np.roll(data["x_hat"], (-4 * sy, -4 * sx), axis=(0, 1))
    return out


def weights(L, weighted):
    w = np.exp(-(L - L.min()) / max(L.std(), 1e-6)) if weighted else np.ones(len(L))
    return w / w.sum()


ens_post, ref_check, loss_check = {}, [], []
for n, i in enumerate(images):
    path = ENS_POST_DIR / f"img{i:05d}.npz"
    if path.exists():
        with np.load(path) as data:
            ens_post[i] = json.loads(str(data["metrics_json"]))
        continue
    y, hr = lr_all[i], load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    members = load_members(i)
    L = {m: lr_loss(x, y) for m, x in members.items()}
    outputs, info = {}, {"L": L, "kept": kept[i]}
    outputs["CSGM, correct label + BP"] = backproject(members[f"r{kept[i]}"], y)
    for name, names in ENSEMBLES.items():
        w = weights(np.array([L[m] for m in names]), name.startswith("E3w"))
        mean_x = sum(wk * members[m] for wk, m in zip(w, names))
        info[name] = {"weights": w.tolist(), "n_eff": float(1 / np.sum(w ** 2)),
                      "mse_raw": mse(hr, mean_x), "lr_loss_raw": lr_loss(mean_x, y)}
        outputs[name + " + BP"] = backproject(mean_x, y)
    info["member_mse"] = {m: mse(hr, x) for m, x in members.items()}
    ens_post[i] = {"info": info, **{m: evaluate(x, hr, y) for m, x in outputs.items()}}
    with np.load(DATA_DIR / "csgm_test_postproc_v1" / f"img{i:05d}.npz") as data:
        ref_check.append(float(np.abs(data["CSGM_correct_label_BP"] - outputs["CSGM, correct label + BP"]).max()))
    loss_check += [abs(L[f"shift{sy}{sx}"] - ens[(i, sy, sx)]["lr_loss_unshifted"]) / L[f"shift{sy}{sx}"] for sy, sx in SHIFTS]
    buffer = io.BytesIO()
    np.savez_compressed(buffer, metrics_json=json.dumps(ens_post[i]),
                        **{m.split(":")[0] + "_BP": outputs[m] for m in ENS_METHODS},
                        CSGM_BP=outputs["CSGM, correct label + BP"])
    path.write_bytes(buffer.getvalue())  # one file per image; a crash leaves earlier images cached
    if (n + 1) % 10 == 0:
        print(f"{n + 1}/{len(images)} images done", flush=True)

if ref_check:
    print(f"max |CSGM + BP here - csgm_test_postproc_v1| (pixels): {max(ref_check):.2e}")
    print(f"max relative |L (CPU, here) - L (GPU, runner)| of shifted members: {max(loss_check):.2e}")
d = max(abs(ens_post[i]["CSGM, correct label + BP"][k] - csgm_post[str(i)]["CSGM, correct label + BP"][k])
        for i in images for k in ("psnr", "lpips"))
print(f"max |CSGM + BP metrics here - notebook 03 Part 2| (PSNR dB / LPIPS): {d:.2e}")
print("max LR loss after BP:", max(ens_post[i][m]["lr_loss"] for i in images for m in ENS_METHODS))
(ENS_POST_DIR / "metrics.json").write_text(json.dumps({str(i): ens_post[i] for i in images}, indent=1))


def v(method, key, band=None):
    # Per-image values; bicubic (+ BP) from notebook 03 Part 2, the rest from here.
    src = {i: csgm_post[str(i)] for i in images} if method.startswith("bicubic") else ens_post
    return np.array([src[i][method]["band_snr_db"][band] if band else src[i][method][key] for i in images])

## Part 1, step 3: primary tests (Addendum A; Holm over 2)

In [ ]:
PRIMARY_TESTS = [("E3w: weighted 5 CSGM runs + BP", "CSGM, correct label + BP", key) for key in ("psnr", "lpips")]
res = [paired_stats(a, b, key, v) for a, b, key in PRIMARY_TESTS]
adj = holm(np.array([r["p"] for r in res]))
lines = ["| # | Comparison (A − B) | Metric | Mean diff. | 95% CI | Wilcoxon p | **Holm p** | A better |",
         "|---:|---|---|---:|---:|---:|---:|---:|"]
for k, ((a, b, key), r, pa) in enumerate(zip(PRIMARY_TESTS, res, adj), start=1):
    f = "{:+.3f}" if key == "lpips" else "{:+.2f}"
    lines.append(f"| {k} | {a} − {b} | {'LPIPS (lower better)' if key == 'lpips' else 'PSNR (dB)'} | {f.format(r['mean'])} | "
                 f"[{f.format(r['lo'])}, {f.format(r['hi'])}] | {r['p']:.1e} | **{pa:.1e}** | {r['better']}/{r['n']} |")
display(Markdown(f"**PRIMARY ANALYSIS (post-hoc test addendum, PROTOCOL §11): {len(images)} test images, correct label**\n\n"
                 + "\n".join(lines)))
if complete:
    (ENS_POST_DIR / "primary.json").write_text(json.dumps(
        [{"comparison": f"{a} − {b}", "metric": key, **{k: float(x) for k, x in r.items()}, "holm_p": float(pa)}
         for (a, b, key), r, pa in zip(PRIMARY_TESTS, res, adj)], indent=1))

## Part 1, step 4: main table and secondary comparisons (exploratory, uncorrected)

In [ ]:
ALL = ["bicubic + BP", "CSGM, correct label + BP"] + ENS_METHODS
lines = ["| Method (+ BP) | runs | Mean PSNR | PSNR (mean MSE) | LPIPS ↓ | Blockiness | " + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |",
         "|---|---:|---:|---:|---:|---:|" + "---:|" * len(BAND_NAMES)]
runs = {"bicubic": 0, "CSGM": 1, "E1": 2, "E2": 4, "E3:": 5, "E3w": 5}
for m in ALL:
    p = v(m, "psnr")
    nr = next(r for k, r in runs.items() if m.startswith(k))
    lines.append(f"| {m} | {nr} | {p.mean():.2f} | {psnr_of_mean_mse(p):.2f} | {v(m, 'lpips').mean():.3f} | {v(m, 'blockiness').mean():.3f} | "
                 + " | ".join(f"{v(m, None, b).mean():.2f}" for b in BAND_NAMES) + " |")
display(Markdown(f"**All methods (+ BP), mean over {len(images)} test images, correct label** (runs = CSGM runs used)\n\n" + "\n".join(lines)))

SECONDARY = [(m, "CSGM, correct label + BP") for m in ENS_METHODS if not m.startswith("E3w")] + \
            [(m, "bicubic + BP") for m in ENS_METHODS] + [("E3w: weighted 5 CSGM runs + BP", "E3: mean of 5 CSGM runs + BP")]
for key, title in (("psnr", "PSNR (dB)"), ("lpips", "LPIPS (lower is better)"), ("blockiness", "Blockiness (lower is better)")):
    f = "{:+.2f}" if key == "psnr" else "{:+.3f}"
    lines = ["| A − B | Mean diff. | 95% CI | Wilcoxon p | A better |", "|---|---:|---:|---:|---:|"]
    for a, b in SECONDARY:
        r = paired_stats(a, b, key, v)
        lines.append(f"| {a} − {b} | {f.format(r['mean'])} | [{f.format(r['lo'])}, {f.format(r['hi'])}] | {r['p']:.1e} | {r['better']}/{r['n']} |")
    display(Markdown(f"**Secondary (exploratory, uncorrected): {title}**\n\n" + "\n".join(lines)))

n_eff = np.array([ens_post[i]["info"]["E3w: weighted 5 CSGM runs"]["n_eff"] for i in images])
Lr = np.array([[ens_post[i]["info"]["L"][m] for m in MEMBERS] for i in images])
display(Markdown(
    f"**E3w weights:** effective number of members 1/Σw² mean {n_eff.mean():.2f} (min {n_eff.min():.2f}, max {n_eff.max():.2f}) of 5.\n\n"
    "**Member LR losses (mean over images):** " + ", ".join(f"{m} {Lr[:, k].mean():.1f}" for k, m in enumerate(MEMBERS))))

## Part 1, step 5: MMSE check (exploratory)

If the members were independent samples from the posterior p(x | y), each would have expected squared error 2·MMSE, and the mean of K of them (1 + 1/K)·MMSE, so averaging K = 5 would lower the error by 10·log₁₀(2/1.2) = **2.22 dB**. Measured here: per image, the members' mean MSE ÷ the MSE of the uniform mean (E3), **before BP**, in dB.

Caveats: CSGM isn't exact posterior sampling (it's a local mode-seeking fit), the members share the generator and the 3 shifted members start from r0's ε and z, so they aren't independent; BP then changes the picture (the + BP gains are in the tables above).

In [ ]:
gain = np.array([10 * np.log10(np.mean(list(ens_post[i]["info"]["member_mse"].values()))
                                / ens_post[i]["info"]["E3: mean of 5 CSGM runs"]["mse_raw"]) for i in images])
half = stats.t.ppf(0.975, len(gain) - 1) * gain.std(ddof=1) / np.sqrt(len(gain))
display(Markdown(f"**MMSE check (E3, before BP):** mean gain {gain.mean():.2f} dB [95% CI {gain.mean() - half:.2f}, {gain.mean() + half:.2f}], "
                 f"median {np.median(gain):.2f}, range {gain.min():.2f} to {gain.max():.2f}; prediction for independent posterior samples: "
                 f"{10 * np.log10(2 / 1.2):.2f} dB. Development (notebook 04 Check 13, after BP): E3 +2.36 dB, E3w +1.60 dB vs CSGM + BP."))
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.hist(gain, bins=20, color="C0", alpha=0.8)
ax.axvline(10 * np.log10(2 / 1.2), color="k", linestyle="--", label="independent-sample prediction (2.22 dB)")
ax.axvline(gain.mean(), color="C3", label=f"measured mean ({gain.mean():.2f} dB)")
ax.set(xlabel="members' mean MSE / ensemble MSE (dB), before BP", ylabel="test images", title="MMSE check, E3 (5 members)")
ax.legend(fontsize=8)
plt.show()

## Part 1, step 6: figures

Scatter of E3w + BP vs CSGM + BP per image, then the first 4 images of the fixed test sample (not chosen by result).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, key, title in ((axes[0], "psnr", "PSNR (dB)"), (axes[1], "lpips", "LPIPS")):
    ref = v("CSGM, correct label + BP", key)
    for k, m in enumerate(["E3: mean of 5 CSGM runs + BP", "E3w: weighted 5 CSGM runs + BP"]):
        ax.scatter(ref, v(m, key), s=12, alpha=0.7, color=f"C{k}", label=m.split(":")[0] + " + BP")
    lo, hi = min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    ax.set(xlabel=f"CSGM + BP: {title}", ylabel=f"ensemble + BP: {title}",
           title=f"{title} per test image" + (" (below diagonal = ensemble better)" if key == "lpips" else " (above = ensemble better)"))
    ax.legend(fontsize=8)
axes[2].boxplot([v(m, "blockiness") for m in ALL], showfliers=False, tick_labels=[m.split(":")[0].replace(", correct label", "") for m in ALL])
axes[2].axhline(1.0, color="gray", linestyle="--", linewidth=1)
axes[2].set(ylabel="blockiness (≈ 1 = no tile seams)", title="Tile-seam blockiness")
axes[2].tick_params(axis="x", rotation=25)
fig.tight_layout()
plt.show()

for i in test_sample[:4]:
    if i not in ens_post:
        continue
    with np.load(ENS_POST_DIR / f"img{i:05d}.npz") as data, np.load(DATA_DIR / "csgm_test_postproc_v1" / f"img{i:05d}.npz") as old:
        shown = [("bicubic + BP", old["bicubic_BP"], csgm_post[str(i)]["bicubic + BP"]),
                 ("CSGM + BP", data["CSGM_BP"], ens_post[i]["CSGM, correct label + BP"]),
                 ("E3 + BP", data["E3_BP"], ens_post[i]["E3: mean of 5 CSGM runs + BP"]),
                 ("E3w + BP", data["E3w_BP"], ens_post[i]["E3w: weighted 5 CSGM runs + BP"])]
    side_by_side(i, load_hr256(DATA_DIR / index_rows[i]["relative_path"]),
                 f"Ground-truth HR, test image {i}\n{class_name(true_label[i])}", shown)

## Part 1 conclusions (100 test images, correct label; post-hoc addendum, PROTOCOL §11)

**Checks passed.** 300/300 shifted runs, all finite, manifest hash `ca2d0ec5e6ff` as frozen. The recomputed CSGM + BP is pixel-identical to `csgm_test_postproc_v1` (max difference 0), its PSNR / LPIPS identical to notebook 03; the CPU LR losses of the shifted members match the runner's GPU values (relative 1.6e-6); LR loss after BP ≤ 2.5e-10.

**Primary (Holm over 2): both significant.**
- E3w + BP − CSGM + BP, PSNR: **+1.57 dB** [1.40, 1.74], better on **100/100** images, Holm p 7.8e-18.
- E3w + BP − CSGM + BP, LPIPS: **−0.032** [−0.040, −0.024], better on 76/100, Holm p 5.7e-10.

**Secondary (exploratory).**
- Uniform E3 does better than the pre-declared E3w on PSNR: +2.13 dB vs CSGM + BP (100/100), LPIPS −0.037; E3w − E3 = −0.56 dB (E3w better on 10/100), LPIPS +0.005 (n.s.). E3w uses on average 2.9 effective members of 5.
- More members, more gain: E1 (2 restarts) +1.01 dB, E2 (r0 + 3 shifts) +1.94, E3 (5) +2.13. Shift members reduce blockiness more than restarts (E2 1.14 vs E1 1.23; CSGM 1.26).
- vs bicubic + BP: every ensemble is still below on PSNR (E3 −0.69 dB, better on 11/100) but clearly better on LPIPS (E3 −0.111, better on 92/100).
- Bands: averaging improves every band, most of all above the LR limit (E3 vs CSGM + BP: high −1.04 → +0.62 dB, very high −4.07 → −1.56 dB).
- **MMSE check:** averaging 5 members lowers the error by **2.33 dB** before BP [2.18, 2.48], close to the 2.22 dB predicted for independent posterior samples.

**Compared with development (notebook 04 Check 13, 6 images):** E3w +1.60 dB / LPIPS −0.029, E3 +2.36 dB / −0.001. The test confirms both PSNR gains and shows a clearer LPIPS gain.

**Deviation (infrastructure only):** the first launch was killed by the Linux OOM killer after 2 saved runs (a concurrent smoke-test process); it resumed with identical settings; the 2 saved runs were kept.

Presentation version: `github/notebooks/03_ensemble_posterior_mean.ipynb`.